# 풍투데이 크롤링 (소프트콘 기준)

## 오늘 목표 : 소프트콘 데이터에서 가져온 bj의 풍투데이 25년1월~26년3월 누적풍/시급풍 수집하기

### 먼저 테스트로 승복님의 100명 중 soop 버튜버의 풍 정보 추출하기

graph TD

    A[1. 원본 데이터 로드] --> B(2. 플랫폼 필터링: SOOP)
    B --> C{3. Channel ID 확인}
    
    C -- ❌ ID 없음 --> D[📁 channel_id_null 결측치 저장]
    C -- 🟢 정상 ID --> E{4. 기존 수집 파일 대조}
    
    E -- 겹침 --> F[수집 패스]
    E -- 안 겹침 --> G((5. 크롤링 타겟 확정))
    
    G --> H[🌐 풍투데이 접속 및 데이터 수집 25.01 ~ 26.03]
    
    H --> I{6. 수집 결과 판별}
    
    I -- 🟢 성공 --> J[📁 done 성공 데이터 저장]
    I -- ❌ 기록 없음 --> K[📁 channel_id_wrong 실패 ID 저장]

In [15]:
import pandas as pd
import os # 폴더 관리를 위해 추가됨
from selenium import webdriver
from selenium.webdriver.chrome.service import Service 
from webdriver_manager.chrome import ChromeDriverManager 
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# ==========================================
# 0. 폴더 구조 자동 생성 (추가됨)
# ==========================================
os.makedirs("MDH/poong_save/channel_id_null", exist_ok=True)
os.makedirs("MDH/poong_save/channel_id_wrong", exist_ok=True)
os.makedirs("MDH/poong_save/done", exist_ok=True)

# ==========================================
# 1. 데이터 불러오기 및 전체 현황 파악
# ==========================================
df = pd.read_csv("final_streamer_data1_100.csv")

# SOOP만
df_soop = df[df['platform'] == 'SOOP'].copy()

# 전체 행 수
total_rows = len(df_soop)

# 스트리머 수
unique_streamers = df_soop['streamer_name'].nunique()

# channel_id를 건드리지 말고 별도 마스크 생성
nan_mask = df_soop['channel_id'].isna()
str_nan_mask = df_soop['channel_id'].astype(str).isin(['', 'None', 'nan', 'NaN']) 

# 결측 포함 마스크
missing_mask = nan_mask | str_nan_mask

# 고유 채널 수 (정상값만)
unique_channels = df_soop.loc[~missing_mask, 'channel_id'].nunique()

# channel_id 없는 스트리머 수
missing_streamers = df_soop.loc[missing_mask, 'streamer_name'].nunique()

# 출력
unique_cnt = df['streamer_name'].nunique()

print("전체 스트리머 수:", unique_cnt)
print(" ")
# 플랫폼별 스트리머 수
platform_cnt = df.groupby('platform')['streamer_name'].nunique()
print(platform_cnt)
print(" ")
# SOOP
print("SOOP 전체 행 수:", total_rows)
print("SOOP 스트리머 수:", unique_streamers)
print("고유 채널 수:", unique_channels)
print(" ")
print("channel_id 없는 스트리머 수:", missing_streamers)
print(df_soop.loc[missing_mask, 'streamer_name'].drop_duplicates())

print("\n" + "="*40)

# ==========================================
# 2. 결측치 행만 모아서 CSV로 따로 저장
# ==========================================
df_missing_channels = df_soop[missing_mask]
# 경로를 channel_id_null 폴더 밑으로 수정함
save_missing_filename = "MDH/poong_save/channel_id_null/soop_missing_channel_id.csv"
df_missing_channels.to_csv(save_missing_filename, index=False, encoding="utf-8-sig")

print(f"channel_id가 없는 데이터 {len(df_missing_channels)}건을 '{save_missing_filename}' 파일로 따로 저장 완료했습니다!")
print("="*40 + "\n")

# ==========================================
# 3. 크롤링 타겟 ID 추출 (결측치 제외 & 기존 데이터와 대조)
# ==========================================
# 결측치가 아닌 "정상 채널 ID"만 필터링 
df_soop_valid = df_soop[~missing_mask]

# 채널 ID와 스트리머 이름을 매핑해둠 
id_to_name = dict(zip(df_soop_valid['channel_id'], df_soop_valid['streamer_name']))
base_ids = set(df_soop_valid['channel_id'].unique())

# 기존에 수집했던 풍투데이 랭킹 데이터 불러오기 (파일명 확인 필요)
df_existing = pd.read_csv("poongtoday_vtuber_rankings.csv")
existing_ids = set(df_existing['channel_id'].dropna().unique())

# 크롤링할 최종 타겟 ID (원본의 정상 ID - 이미 수집한 ID)
target_ids = list(base_ids - existing_ids)

print(f"✅ 결측치 제외 SOOP 버튜버: {len(base_ids)}명")
print(f"✅ 기존에 수집 완료된 버튜버: {len(existing_ids)}명")
print(f"🚀 새롭게 크롤링할 타겟 버튜버: {len(target_ids)}명\n")

# ==========================================
# 4. 크롤링 실행 (타겟이 있을 경우에만 작동)
# ==========================================
if len(target_ids) > 0:
    print(f"크롤링 시작! 총 {len(target_ids)}명의 데이터를 수집합니다.")
    options = Options()
    options.add_argument("start-maximized")
    
    # === OS 에러를 해결하는 핵심 부분 ===
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    all_results = []
    wrong_id_list = [] # 기록 없는 ID를 담을 빈 리스트 추가

    for channel_id in target_ids:
        streamer_name = id_to_name.get(channel_id, "Unknown") 
        print(f"[{streamer_name} ({channel_id})] 수집 시작...")
        
        url = f"https://poong.today/broadcast/{channel_id}"
        driver.get(url)
        time.sleep(4)
        
        has_data = False # 이 스트리머의 데이터가 한 건이라도 수집되었는지 체크하는 변수 추가
        
        for i in range(20): 
            time.sleep(1.5) 
            try:
                year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
                month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
                
                # 수집 조건: 25년 1월 ~ 26년 3월
                if (year == 2025) or (year == 2026 and month <= 3):
                    lis = driver.find_elements(By.TAG_NAME, "li")
                    total_star, hourly_star = None, None
                    
                    for li in lis:
                        text = li.text
                        if "누적 별풍선" in text:
                            total_star = li.find_element(By.TAG_NAME, "span").text
                        if "시급(풍)" in text:
                            hourly_star = li.find_element(By.TAG_NAME, "span").text
                    
                    total_star = int(total_star.replace(",", "")) if total_star else 0
                    hourly_star = int(hourly_star.replace(",", "")) if hourly_star else 0
                    
                    all_results.append({
                        "name": streamer_name,
                        "channel_id": channel_id, 
                        "year": year,
                        "month": month,
                        "total_star": total_star,
                        "hourly_star": hourly_star
                    })
                    has_data = True # 데이터를 수집했으므로 True로 변경
                    print(f"   - {year}-{month:02d} 수집 완료")
                
                elif year < 2025:
                    print(f"   - 2024년 이전 데이터 도달. 다음 채널로 이동합니다.")
                    break
                    
                prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
                prev_btn.click()
                
            except Exception as e:
                print(f"   - 더 이상의 과거 기록이 없습니다.")
                break
        
        # 20번의 반복(또는 예외처리 탈출)이 끝난 후, 수집된 데이터가 없다면 wrong_id_list에 추가
        if not has_data:
            print(f"   ⚠️ {streamer_name}: 수집된 기록 없음 (Wrong ID 분류)")
            wrong_id_list.append({"name": streamer_name, "channel_id": channel_id})
                
        print("-" * 30)

    driver.quit()

    # ==========================================
    # 5. 결과 저장 (rank 없음)
    # ==========================================
    # [성공 데이터 저장] 경로를 done 폴더 밑으로 수정
    if all_results:
        df_new_data = pd.DataFrame(all_results)
        df_new_data = df_new_data[['name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]
        save_filename = "MDH/poong_save/done/new_vtuber_data_no_rank.csv"
        df_new_data.to_csv(save_filename, index=False, encoding="utf-8-sig")
        print(f"🎉 크롤링 완료! '{save_filename}' 파일이 생성되었습니다.")

    # [수집 실패(Wrong ID) 데이터 저장] 새로 추가된 부분
    if wrong_id_list:
        df_wrong = pd.DataFrame(wrong_id_list)
        wrong_filename = "MDH/poong_save/channel_id_wrong/wrong_channel_id.csv"
        df_wrong.to_csv(wrong_filename, index=False, encoding="utf-8-sig")
        print(f"⚠️ 기록 없는 ID 데이터 {len(wrong_id_list)}건이 '{wrong_filename}'에 저장되었습니다.")
    
else:
    print("🎉 새롭게 수집할 대상이 없습니다! (모두 이미 수집됨)")

전체 스트리머 수: 100
 
platform
CHZZK      58
SOOP       40
youtube     2
Name: streamer_name, dtype: int64
 
SOOP 전체 행 수: 14624
SOOP 스트리머 수: 40
고유 채널 수: 34
 
channel_id 없는 스트리머 수: 6
21730      단즈
22979      띵귤
23348     라로시
24445    마이곰이
28734     여르미
32591      한결
Name: streamer_name, dtype: str

channel_id가 없는 데이터 2212건을 'MDH/poong_save/channel_id_null/soop_missing_channel_id.csv' 파일로 따로 저장 완료했습니다!

✅ 결측치 제외 SOOP 버튜버: 34명
✅ 기존에 수집 완료된 버튜버: 945명
🚀 새롭게 크롤링할 타겟 버튜버: 16명

크롤링 시작! 총 16명의 데이터를 수집합니다.
[아이네♪ (inehine)] 수집 시작...
   - 2026-03 수집 완료
   - 2026-02 수집 완료
   - 2026-01 수집 완료
   - 2025-12 수집 완료
   - 2025-11 수집 완료
   - 2025-10 수집 완료
   - 2025-09 수집 완료
   - 2025-08 수집 완료
   - 2025-07 수집 완료
   - 2025-06 수집 완료
   - 2025-05 수집 완료
   - 2025-04 수집 완료
   - 2025-03 수집 완료
   - 2025-02 수집 완료
   - 2025-01 수집 완료
   - 2024년 이전 데이터 도달. 다음 채널로 이동합니다.
------------------------------
[사이다! (42dadada)] 수집 시작...
   - 2026-03 수집 완료
   - 2026-02 수집 완료
   - 2026-01 수집 완료
   - 2025-12 수집 완료
   - 2025-11 수집 완료
   

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=147.0.7727.56)
Stacktrace:
	chromedriver!GetHandleVerifier [0xeeb373+10553]
	chromedriver!GetHandleVerifier [0xeeb4a4+10684]
	chromedriver!(No symbol) [0xcf1e10]
	chromedriver!(No symbol) [0xcd0923]
	chromedriver!(No symbol) [0xd63bcb]
	chromedriver!(No symbol) [0xd7a3a9]
	chromedriver!(No symbol) [0xd5d606]
	chromedriver!(No symbol) [0xd30039]
	chromedriver!(No symbol) [0xd30df4]
	chromedriver!GetHandleVerifier [0x116ddb4+292f94]
	chromedriver!GetHandleVerifier [0x116937d+28e55d]
	chromedriver!GetHandleVerifier [0x1188215+2ad3f5]
	chromedriver!GetHandleVerifier [0xf04c18+29df8]
	chromedriver!GetHandleVerifier [0xf0c6fd+318dd]
	chromedriver!GetHandleVerifier [0xef3c78+18e58]
	chromedriver!GetHandleVerifier [0xef3e42+19022]
	chromedriver!GetHandleVerifier [0xedd45f+263f]
	KERNEL32!BaseThreadInitThunk [0x75ca5d49+19]
	ntdll!RtlInitializeExceptionChai

# 실전! (??)

## 최종 소프트콘 병합 데이터로 실행!

shape : (2293860, 11)

파일 크기: 192.70 MB

In [28]:
import pandas as pd
import os 
from selenium import webdriver
from selenium.webdriver.chrome.service import Service 
from webdriver_manager.chrome import ChromeDriverManager 
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
suffix = "1p_122p"
# ==========================================
# 0. 폴더 구조 자동 생성 
# ==========================================
os.makedirs("poong_save/channel_id_null", exist_ok=True)
os.makedirs("poong_save/channel_id_wrong", exist_ok=True)
os.makedirs("poong_save/done", exist_ok=True)

# ==========================================
# 1. 데이터 불러오기 및 전체 현황 파악
# ==========================================
df = pd.read_csv("softcone_streamer_dataset _clean.csv")

# SOOP만
df_soop = df[df['platform'] == 'SOOP'].copy()

# 전체 행 수
total_rows = len(df_soop)

# 스트리머 수
unique_streamers = df_soop['streamer_name'].nunique()

# channel_id를 건드리지 말고 별도 마스크 생성
nan_mask = df_soop['channel_id'].isna()
str_nan_mask = df_soop['channel_id'].astype(str).isin(['', 'None', 'nan', 'NaN']) 

# 결측 포함 마스크
missing_mask = nan_mask | str_nan_mask

# 고유 채널 수 (정상값만)
unique_channels = df_soop.loc[~missing_mask, 'channel_id'].nunique()

# channel_id 없는 스트리머 수
missing_streamers = df_soop.loc[missing_mask, 'streamer_name'].nunique()

# 출력
unique_cnt = df['streamer_name'].nunique()

print("전체 스트리머 수:", unique_cnt)
print(" ")
# 플랫폼별 스트리머 수
platform_cnt = df.groupby('platform')['streamer_name'].nunique()
print(platform_cnt)
print(" ")
# SOOP
print("SOOP 전체 행 수:", total_rows)
print("SOOP 스트리머 수:", unique_streamers)
print("고유 채널 수:", unique_channels)
print(" ")
print("channel_id 없는 스트리머 수:", missing_streamers)
print(df_soop.loc[missing_mask, 'streamer_name'].drop_duplicates())

print("\n" + "="*40)

# ==========================================
# 2. 결측치 행만 모아서 CSV로 따로 저장
# ==========================================
df_missing_channels = df_soop[missing_mask]
# 경로를 channel_id_null 폴더 밑으로 수정함
save_missing_filename = f"poong_save/channel_id_null/soop_missing_channel_id_{suffix}.csv"
df_missing_channels.to_csv(save_missing_filename, index=False, encoding="utf-8-sig")

print(f"channel_id가 없는 데이터 {len(df_missing_channels)}건을 '{save_missing_filename}' 파일로 따로 저장 완료했습니다!")
print("="*40 + "\n")

# ==========================================
# 3. 크롤링 타겟 ID 추출 (결측치 제외 & 기존 데이터와 대조)
# ==========================================
# 결측치가 아닌 "정상 채널 ID"만 필터링 
df_soop_valid = df_soop[~missing_mask]

# 채널 ID와 스트리머 이름을 매핑해둠 
id_to_name = dict(zip(df_soop_valid['channel_id'], df_soop_valid['streamer_name']))
base_ids = set(df_soop_valid['channel_id'].unique())

# 기존에 수집했던 풍투데이 랭킹 데이터 불러오기 
df_existing = pd.read_csv("poongtoday_vtuber_rankings.csv")
existing_ids = set(df_existing['channel_id'].dropna().unique())

# 크롤링할 최종 타겟 ID (원본의 정상 ID - 이미 수집한 ID)
target_ids = list(base_ids - existing_ids)

print(f"결측치 제외 SOOP 버튜버: {len(base_ids)}명")
print(f"기존에 수집 완료된 버튜버: {len(existing_ids)}명")
print("이미 수집된 스트리머 수:", len(base_ids & existing_ids))
print(f"새롭게 크롤링할 타겟 버튜버: {len(target_ids)}명\n")

# ==========================================
# 4. 크롤링 실행 (타겟이 있을 경우에만 작동)
# ==========================================
if len(target_ids) > 0:
    print(f"크롤링 시작! 총 {len(target_ids)}명의 데이터를 수집합니다.")
    options = Options()
    options.add_argument("start-maximized")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    all_results = []
    wrong_id_list = [] # 기록 없는 ID를 담을 빈 리스트 추가

    for channel_id in target_ids:
        streamer_name = id_to_name.get(channel_id, "Unknown") 
        print(f"[{streamer_name} ({channel_id})] 수집 시작...")
        
        url = f"https://poong.today/broadcast/{channel_id}"
        driver.get(url)
        time.sleep(2)
        
        has_data = False 
        
        for i in range(20): 
            time.sleep(1.5) 
            try:
                year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
                month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
                
                # 수집 조건: 25년 1월 ~ 26년 3월
                if (year == 2025) or (year == 2026 and month <= 3):
                    lis = driver.find_elements(By.TAG_NAME, "li")
                    total_star, hourly_star = None, None
                    
                    for li in lis:
                        text = li.text
                        if "누적 별풍선" in text:
                            total_star = li.find_element(By.TAG_NAME, "span").text
                        if "시급(풍)" in text:
                            hourly_star = li.find_element(By.TAG_NAME, "span").text
                    
                    total_star = int(total_star.replace(",", "")) if total_star else 0
                    hourly_star = int(hourly_star.replace(",", "")) if hourly_star else 0
                    
                    all_results.append({
                        "name": streamer_name,
                        "channel_id": channel_id, 
                        "year": year,
                        "month": month,
                        "total_star": total_star,
                        "hourly_star": hourly_star
                    })
                    has_data = True # 데이터를 수집했으므로 True로 변경
                    print(f"   - {year}-{month:02d} 수집 완료")
                
                elif year < 2025:
                    print(f"   - 2024년 이전 데이터 도달. 다음 채널로 이동합니다.")
                    break
                    
                prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
                prev_btn.click()
                
            except Exception as e:
                print(f"   - 더 이상의 과거 기록이 없습니다.")
                break
        
        # 데이터가 수집되지 않는다면 따로 csv에 저장
        if not has_data:
            print(f"   ⚠️ {streamer_name}: 수집된 기록 없음 (Wrong ID 분류)")
            wrong_id_list.append({"name": streamer_name, "channel_id": channel_id})
                
        print("-" * 30)

    driver.quit()

    # ==========================================
    # 5. 결과 저장 
    # ==========================================
    # [성공 데이터 저장] 경로를 done 폴더 밑으로 수정
    if all_results:
        df_new_data = pd.DataFrame(all_results)
        df_new_data = df_new_data[['name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]
        save_filename = f"poong_save/done/new_vtuber_data_{suffix}.csv"
        df_new_data.to_csv(save_filename, index=False, encoding="utf-8-sig")
        print(f"크롤링 완료! '{save_filename}' 파일이 생성되었습니다.")

    if wrong_id_list:
        df_wrong = pd.DataFrame(wrong_id_list)
        wrong_filename = f"poong_save/channel_id_wrong/wrong_channel_id_{suffix}.csv"
        df_wrong.to_csv(wrong_filename, index=False, encoding="utf-8-sig")
        print(f"⚠️ 기록 없는 ID 데이터 {len(wrong_id_list)}건이 '{wrong_filename}'에 저장되었습니다.")

else:
    print("새롭게 수집할 대상이 없습니다! (모두 이미 수집됨)")

전체 스트리머 수: 11679
 
platform
CHZZK      9174
CIME        528
SOOP       2387
TWITCH        4
youtube       3
Name: streamer_name, dtype: int64
 
SOOP 전체 행 수: 531221
SOOP 스트리머 수: 2387
고유 채널 수: 2387
 
channel_id 없는 스트리머 수: 0
Series([], Name: streamer_name, dtype: str)

channel_id가 없는 데이터 0건을 'poong_save/channel_id_null/soop_missing_channel_id_1p_122p.csv' 파일로 따로 저장 완료했습니다!

결측치 제외 SOOP 버튜버: 2387명
기존에 수집 완료된 버튜버: 945명
이미 수집된 스트리머 수: 941
새롭게 크롤링할 타겟 버튜버: 1446명

크롤링 시작! 총 1446명의 데이터를 수집합니다.
[도이리_ (hh2154)] 수집 시작...
   - 2026-03 수집 완료
   - 더 이상의 과거 기록이 없습니다.
------------------------------
[딸기크래커 (shhm9792)] 수집 시작...


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0xeeb373+10553]
	chromedriver!GetHandleVerifier [0xeeb4a4+10684]
	chromedriver!(No symbol) [0xcf1c4e]
	chromedriver!(No symbol) [0xd2f2c5]
	chromedriver!(No symbol) [0xd5d726]
	chromedriver!(No symbol) [0xd58c62]
	chromedriver!(No symbol) [0xd58281]
	chromedriver!(No symbol) [0xcc501e]
	chromedriver!(No symbol) [0xcc55be]
	chromedriver!(No symbol) [0xcc5a9d]
	chromedriver!GetHandleVerifier [0x116ddb4+292f94]
	chromedriver!GetHandleVerifier [0x116937d+28e55d]
	chromedriver!GetHandleVerifier [0x1188215+2ad3f5]
	chromedriver!GetHandleVerifier [0xf04c18+29df8]
	chromedriver!GetHandleVerifier [0xf0c6fd+318dd]
	chromedriver!(No symbol) [0xcc4b99]
	chromedriver!(No symbol) [0xcc41e0]
	chromedriver!GetHandleVerifier [0x12d2bef+3f7dcf]
	KERNEL32!BaseThr

# SAVE 버전 만들기

In [47]:
import pandas as pd
import os 
from selenium import webdriver
from selenium.webdriver.chrome.service import Service 
from webdriver_manager.chrome import ChromeDriverManager 
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException 
import time

suffix = "softcone_clean"

# ==========================================
# 0. 폴더 구조 자동 생성 
# ==========================================
os.makedirs("poong_save/channel_id_null", exist_ok=True)
os.makedirs("poong_save/channel_id_wrong", exist_ok=True)
os.makedirs("poong_save/done", exist_ok=True)

# 저장될 파일 경로 미리 지정
success_file = f"poong_save/done/new_vtuber_data_{suffix}.csv"
wrong_file = f"poong_save/channel_id_wrong/wrong_channel_id_{suffix}.csv"
save_missing_filename = f"poong_save/channel_id_null/soop_missing_channel_id_{suffix}.csv"

# ==========================================
# 1. 데이터 불러오기 및 전체 현황 파악
# ==========================================
df = pd.read_csv("softcone_streamer_dataset_clean.csv") 

# SOOP만
df_soop = df[df['platform'] == 'SOOP'].copy()

# 전체 행 수
total_rows = len(df_soop)

# 스트리머 수
unique_streamers = df_soop['streamer_name'].nunique()

# channel_id를 건드리지 말고 별도 마스크 생성
nan_mask = df_soop['channel_id'].isna()
str_nan_mask = df_soop['channel_id'].astype(str).isin(['', 'None', 'nan', 'NaN']) 

# 결측 포함 마스크
missing_mask = nan_mask | str_nan_mask

# 고유 채널 수 (정상값만)
unique_channels = df_soop.loc[~missing_mask, 'channel_id'].nunique()

# channel_id 없는 스트리머 수
missing_streamers = df_soop.loc[missing_mask, 'streamer_name'].nunique()

# 출력 (다현 님의 오리지널 print문 100% 유지)
unique_cnt = df['streamer_name'].nunique()

print("전체 스트리머 수:", unique_cnt)
print(" ")
# 플랫폼별 스트리머 수
platform_cnt = df.groupby('platform')['streamer_name'].nunique()
print(platform_cnt)
print(" ")
# SOOP
print("SOOP 전체 행 수:", total_rows)
print("SOOP 스트리머 수:", unique_streamers)
print("고유 채널 수:", unique_channels)
print(" ")
print("channel_id 없는 스트리머 수:", missing_streamers)
print(df_soop.loc[missing_mask, 'streamer_name'].drop_duplicates())

print("\n" + "="*40)

# ==========================================
# 2. 결측치 행만 모아서 CSV로 따로 저장
# ==========================================
df_missing_channels = df_soop[missing_mask]
df_missing_channels.to_csv(save_missing_filename, index=False, encoding="utf-8-sig")

# (다현 님의 오리지널 print문 100% 유지)
print(f"channel_id가 없는 데이터 {len(df_missing_channels)}건을 '{save_missing_filename}' 파일로 따로 저장 완료했습니다!")
print("="*40 + "\n")

# ==========================================
# 3. 크롤링 타겟 ID 추출 (결측치 제외 & 기존 데이터와 대조)
# ==========================================
# 결측치가 아닌 "정상 채널 ID"만 필터링 
df_soop_valid = df_soop[~missing_mask]

# 채널 ID와 스트리머 이름을 매핑해둠 
id_to_name = dict(zip(df_soop_valid['channel_id'], df_soop_valid['streamer_name']))
base_ids = set(df_soop_valid['channel_id'].unique())

existing_ids = set()

# 기존에 수집했던 풍투데이 랭킹 데이터 불러오기 
if os.path.exists("poongtoday_vtuber_rankings.csv"):
    df_existing = pd.read_csv("poongtoday_vtuber_rankings.csv")
    existing_ids.update(df_existing['channel_id'].dropna().unique())

# 중간에 멈췄다가 다시 켤 때, 이미 저장된 결과 파일을 읽어서 대상에서 제외함 (무적 방어 1)
for f in [success_file, wrong_file]:
    if os.path.exists(f):
        try:
            df_check = pd.read_csv(f)
            if not df_check.empty:
                existing_ids.update(df_check['channel_id'].dropna().unique())
        except: pass

# 크롤링할 최종 타겟 ID
target_ids = list(base_ids - existing_ids)

print(f"결측치 제외 SOOP 버튜버: {len(base_ids)}명")
print(f"기존에 수집 완료된 버튜버: {len(existing_ids)}명")
print("이미 수집된 스트리머 수:", len(base_ids & existing_ids))
print(f"새롭게 크롤링할 타겟 버튜버: {len(target_ids)}명\n")

# ==========================================
# 4. 크롤링 실행
# ==========================================
if len(target_ids) > 0:
    print(f"크롤링 시작! 총 {len(target_ids)}명의 데이터를 수집합니다.")
    options = Options()
    options.add_argument("start-maximized")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    try:
        for channel_id in target_ids:
            streamer_name = id_to_name.get(channel_id, "Unknown") 
            print(f"[{streamer_name} ({channel_id})] 수집 시작...")
            
            url = f"https://poong.today/broadcast/{channel_id}"
            driver.get(url)
            time.sleep(2)
            
            temp_results = []
            has_data = False 
            
            try:
                for i in range(20): 
                    time.sleep(1.5) 
                    year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
                    month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
                    
                    if (year == 2025) or (year == 2026 and month <= 3):
                        lis = driver.find_elements(By.TAG_NAME, "li")
                        total_star, hourly_star = 0, 0
                        
                        for li in lis:
                            text = li.text
                            if "누적 별풍선" in text:
                                total_star = int(li.find_element(By.TAG_NAME, "span").text.replace(",", ""))
                            if "시급(풍)" in text:
                                hourly_star = int(li.find_element(By.TAG_NAME, "span").text.replace(",", ""))
                        
                        temp_results.append({
                            "name": streamer_name, "channel_id": channel_id, 
                            "year": year, "month": month,
                            "total_star": total_star, "hourly_star": hourly_star
                        })
                        has_data = True 
                        print(f"   - {year}-{month:02d} 수집 완료")
                    
                    elif year < 2025:
                        print(f"   - 2024년 이전 데이터 도달. 다음 채널로 이동합니다.") 
                        break
                        
                    driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow").click()
                    
            except NoSuchElementException:
                print(f"   - 더 이상의 과거 기록이 없습니다.") 
                
            except Exception as e:
                # 무적 방어 2: 튕겼을 때 저장 중단
                print(f"\n🚨 [돌발 상황 감지] 브라우저가 닫혔거나 에러가 발생했습니다!")
                print(f"'{streamer_name}'의 불완전한 데이터를 저장하지 않고 즉시 중단합니다.")
                raise e 
            
            # 한 명의 정상적인 수집이 끝날 때마다 CSV에 즉시 저장
            if has_data:
                df_new_data = pd.DataFrame(temp_results)
                header = not os.path.exists(success_file) 
                df_new_data.to_csv(success_file, mode='a', index=False, header=header, encoding="utf-8-sig")
            else:
                print(f"   ⚠️ {streamer_name}: 수집된 기록 없음 (Wrong ID 분류)")
                df_wrong = pd.DataFrame([{"name": streamer_name, "channel_id": channel_id}])
                header = not os.path.exists(wrong_file)
                df_wrong.to_csv(wrong_file, mode='a', index=False, header=header, encoding="utf-8-sig")
                
            print("-" * 30)

    finally:
        driver.quit() 
        print("이번 크롤링 세션이 종료되었습니다.")

else:
    print("새롭게 수집할 대상이 없습니다! (모두 이미 수집됨)")

전체 스트리머 수: 11677
 
platform
CHZZK      9174
CIME        528
SOOP       2385
TWITCH        4
youtube       3
Name: streamer_name, dtype: int64
 
SOOP 전체 행 수: 531221
SOOP 스트리머 수: 2385
고유 채널 수: 2385
 
channel_id 없는 스트리머 수: 0
Series([], Name: streamer_name, dtype: str)

channel_id가 없는 데이터 0건을 'poong_save/channel_id_null/soop_missing_channel_id_softcone_clean.csv' 파일로 따로 저장 완료했습니다!

결측치 제외 SOOP 버튜버: 2385명
기존에 수집 완료된 버튜버: 2385명
이미 수집된 스트리머 수: 2385
새롭게 크롤링할 타겟 버튜버: 0명

새롭게 수집할 대상이 없습니다! (모두 이미 수집됨)


### 풍투데이 집게 스트리머수 : 941 (945에서 안겹치는 4명 제외)

### 크롤링 필요한 스트리머 수 : 1446명

### (새로 크롤링한 스트리머 수 : 1362명) + (집계되지 않은 스트리머 수 : 82명) = 1444명 확인


### 총 2387명

In [2]:
import pandas as pd

try:
    # 1. 파일 불러오기
    poong_df = pd.read_csv('poongtoday_vtuber_rankings.csv')
    softcone_clean_df = pd.read_csv('poong_save/done/new_vtuber_data_softcone_clean.csv')
    wrong_id_df = pd.read_csv('poong_save/channel_id_wrong/wrong_channel_id_softcone_clean.csv')

    # 2. 각 파일별 스트리머 수 확인 (channel_id 기준 중복 제거)
    cnt_poong = poong_df['channel_id'].nunique()
    cnt_softcone = softcone_clean_df['channel_id'].nunique()
    cnt_wrong = wrong_id_df['channel_id'].nunique()

    print(f"1. 풍투데이 랭킹 고유 채널 수: {cnt_poong}개")
    print(f"2. 소프트콘 클린 데이터 고유 채널 수: {cnt_softcone}개")
    print(f"3. 채널 ID 오류 데이터 고유 채널 수: {cnt_wrong}개")

    # 3. SOOP 전체 유니크 인원 확인 (소프트콘 클린 + 오류 데이터)
    soop_total_set = set(softcone_clean_df['channel_id']) | set(wrong_id_df['channel_id'])
    print(f"-> SOOP 전체 고유 채널 (A): {len(soop_total_set)}개 (기대값: 1444)")

    # 4. 풍투데이와 SOOP 전체의 합집합 확인
    poong_set = set(poong_df['channel_id'])
    final_combined_set = soop_total_set | poong_set
    
    # 5. 교집합(겹치는 인원) 확인
    intersection = soop_total_set & poong_set

    print("="*40)
    print(f"교집합(겹치는 채널): {len(intersection)}개 (기대값: 0)")
    print(f"최종 합계(Unique): {len(final_combined_set)}개")
    print("="*40)

    if len(final_combined_set) == 2385:
        print("검증 성공! 채널 ID 기준으로도 데이터가 정확히 2385개로 확인되었습니다.")
    else:
        print(f"검증 필요: 현재 {len(final_combined_set)}개입니다. 차이를 확인해보세요.")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다: {e}")

1. 풍투데이 랭킹 고유 채널 수: 941개
2. 소프트콘 클린 데이터 고유 채널 수: 1362개
3. 채널 ID 오류 데이터 고유 채널 수: 82개
-> SOOP 전체 고유 채널 (A): 1444개 (기대값: 1444)
교집합(겹치는 채널): 0개 (기대값: 0)
최종 합계(Unique): 2385개
검증 성공! 채널 ID 기준으로도 데이터가 정확히 2385개로 확인되었습니다.


## 그러면 wrong_channel_id로 분류된 친구들은 어떻게 풍정보를 채울까?

- 82개

- 평균 시청자 수로 스트리머를 그룹화한 뒤, 각 그룹의 평균 별풍을 계산한다.
- 그리고 별풍 데이터가 없는 스트리머는 자신이 속한 시청자 그룹의 평균값으로 채운다. (SOOP 에서 직접 검색해보았을 때 활발하게 활동하는 스트리머만 해당)
- 뜸하거나 계정정보가 나오지 않으면 모두 0처리

# 잠깐 홀드 

지금 상황 : wrong_id 데이터를 메꾸려고 파악하는 중에 홀드됨.....

어차피 평균 시청자수 구하려면 합산 파일이 나와야 함


그럼 일단 원래풍 (941)이랑 done이랑 합쳐놓고

나중에 wrong_id 82명 메꿔지면 그걸 붙이자


#### 확인 결과 poong투데이(941명 데이터에서2명의정보가 wrong_id로 판단)

poong_df : 945 -> 941 -> 939

softcone_clean_df : 1362

wrong_id_df : 84 -> 82 -> 84

In [30]:
poong_df = pd.read_csv('poongtoday_vtuber_rankings.csv')
softcone_clean_df = pd.read_csv('poong_save/done/new_vtuber_data_softcone_clean.csv')
wrong_id_df = pd.read_csv('poong_save/channel_id_wrong/wrong_channel_id_softcone_clean.csv')


In [31]:
cnt_poong = poong_df['channel_id'].nunique()
cnt_softcone = softcone_clean_df['channel_id'].nunique()
cnt_wrong = wrong_id_df['channel_id'].nunique()
print(cnt_poong,cnt_softcone,cnt_wrong)

939 1362 84


In [36]:
print(poong_df.head())
print(" ")
print(poong_df.info())

   rank name channel_id    year  month  total_star  hourly_star
0     1  릴파♬  lilpa0309  2026.0    3.0    813222.0       6847.0
1     1  릴파♬  lilpa0309  2026.0    2.0    375888.0       4903.0
2     1  릴파♬  lilpa0309  2026.0    1.0    263410.0       2398.0
3     1  릴파♬  lilpa0309  2025.0   12.0    356332.0       4373.0
4     1  릴파♬  lilpa0309  2025.0   11.0    315947.0       4662.0
 
<class 'pandas.DataFrame'>
RangeIndex: 14085 entries, 0 to 14084
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   rank         14085 non-null  int64  
 1   name         14085 non-null  str    
 2   channel_id   14085 non-null  str    
 3   year         14085 non-null  float64
 4   month        14085 non-null  float64
 5   total_star   14085 non-null  float64
 6   hourly_star  14085 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 770.4 KB
None


In [ ]:
print(softcone_clean_df.head())
print(" ")
print(softcone_clean_df.info())

    name  channel_id  year  month  total_star  hourly_star
0  바난나우유  tyyobich81  2026      3           0            0
1  바난나우유  tyyobich81  2026      2           0            0
2  바난나우유  tyyobich81  2026      1           0            0
3  바난나우유  tyyobich81  2025     12           0            0
4  바난나우유  tyyobich81  2025     11           0            0
 
<class 'pandas.DataFrame'>
RangeIndex: 20430 entries, 0 to 20429
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         20430 non-null  str  
 1   channel_id   20430 non-null  str  
 2   year         20430 non-null  int64
 3   month        20430 non-null  int64
 4   total_star   20430 non-null  int64
 5   hourly_star  20430 non-null  int64
dtypes: int64(4), str(2)
memory usage: 957.8 KB
None


In [43]:
print(poong_df.columns)
print(softcone_clean_df.columns)
print("")
print("미리 긁은 풍투데이 컬럼 개수 :",len(poong_df.columns))
print("새로 긁은 풍투데이 컬럼 개수 :",len(softcone_clean_df.columns))

Index(['rank', 'name', 'channel_id', 'year', 'month', 'total_star',
       'hourly_star'],
      dtype='str')
Index(['name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star'], dtype='str')

미리 긁은 풍투데이 컬럼 개수 : 7
새로 긁은 풍투데이 컬럼 개수 : 6


poong_df에서 rank 컬럼 버리기

In [44]:
poong_df.drop(columns=['rank'], inplace=True)

print("poong_df 컬럼:", poong_df.columns.tolist())
print("softcone_clean_df 컬럼:", softcone_clean_df.columns.tolist())

poong_df 컬럼: ['name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']
softcone_clean_df 컬럼: ['name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']


데이터타입 맞추기

In [45]:
# poong_df의 컬럼들을 정수형으로 바꾸기
cols_to_int = ['year', 'month', 'total_star', 'hourly_star']
poong_df[cols_to_int] = poong_df[cols_to_int].astype(int)

print(poong_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 14085 entries, 0 to 14084
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         14085 non-null  str  
 1   channel_id   14085 non-null  str  
 2   year         14085 non-null  int64
 3   month        14085 non-null  int64
 4   total_star   14085 non-null  int64
 5   hourly_star  14085 non-null  int64
dtypes: int64(4), str(2)
memory usage: 660.4 KB
None


합치기

In [48]:
import pandas as pd

combined_df = pd.concat([poong_df, softcone_clean_df], ignore_index=True)

print(f"poong_df 행 수: {len(poong_df)}")
print(f"softcone_clean_df 행 수: {len(softcone_clean_df)}")
print(f"합쳐진 전체 행 수: {len(combined_df)}")

duplicate_count = combined_df.duplicated(subset=['channel_id', 'year', 'month']).sum()
print(f"\n중복된 데이터(ID/년/월 기준) 개수: {duplicate_count}")


print("\n--- 최종 데이터프레임 정보 ---")
print(combined_df.info())

poong_df 행 수: 14085
softcone_clean_df 행 수: 20430
합쳐진 전체 행 수: 34515

중복된 데이터(ID/년/월 기준) 개수: 0

--- 최종 데이터프레임 정보 ---
<class 'pandas.DataFrame'>
RangeIndex: 34515 entries, 0 to 34514
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         34515 non-null  str  
 1   channel_id   34515 non-null  str  
 2   year         34515 non-null  int64
 3   month        34515 non-null  int64
 4   total_star   34515 non-null  int64
 5   hourly_star  34515 non-null  int64
dtypes: int64(4), str(2)
memory usage: 1.6 MB
None


In [49]:
print(combined_df.head())
print(" ")
print(combined_df.info())

  name channel_id  year  month  total_star  hourly_star
0  릴파♬  lilpa0309  2026      3      813222         6847
1  릴파♬  lilpa0309  2026      2      375888         4903
2  릴파♬  lilpa0309  2026      1      263410         2398
3  릴파♬  lilpa0309  2025     12      356332         4373
4  릴파♬  lilpa0309  2025     11      315947         4662
 
<class 'pandas.DataFrame'>
RangeIndex: 34515 entries, 0 to 34514
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         34515 non-null  str  
 1   channel_id   34515 non-null  str  
 2   year         34515 non-null  int64
 3   month        34515 non-null  int64
 4   total_star   34515 non-null  int64
 5   hourly_star  34515 non-null  int64
dtypes: int64(4), str(2)
memory usage: 1.6 MB
None


풍투데이 + done = combined_df

combined_df의 고유 스트리머 수 : 2301개

In [50]:
cnt_combined_df = combined_df['channel_id'].nunique()
print(cnt_combined_df)

2301


In [55]:
import os

folder_path = 'poong_save/result'
file_name = 'poongtoday_combined_data1.csv'
full_path = os.path.join(folder_path, file_name)

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"폴더를 생성했습니다: {folder_path}")

combined_df.to_csv(full_path, index=False, encoding='utf-8-sig')

print(f"파일 저장 완료: {full_path}")

파일 저장 완료: poong_save/result\poongtoday_combined_data1.csv


In [58]:
a = pd.read_csv("poong_save/result/poongtoday_combined_data1.csv")
cnt_a = a['channel_id'].nunique()
print("통합파일의 스트리머 수",cnt_a)

통합파일의 스트리머 수 2301


### 아직 wrong_id 처리 후 통합이 필요하다!!!!